In [ ]:
from pathlib import Path
import uproot
import pandas as pd
import numpy as np
import warnings
import json
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore', 'DataFrame is highly fragmented')

In [148]:
root_file = Path('/eos/user/a/anunezde/Z_OUTPUT_eos/Disc_Study_New/LLR_odd/results/ggHH_kl_1_kt_1_bbww_dl_2022.root')
with uproot.open(root_file) as upfile:
    tree = upfile['SL_4j_resolved']
    df = tree.arrays(library="pd")
cols_to_keep = [col for col in df.columns if '_x' not in col and '_vs_' not in col] # col not in ['event', 'genWeight']
df = df[cols_to_keep]
print('Total number of events:', len(df))

Total number of events: 1783


In [155]:
cols_with_pos_inf = df.columns[df.apply(lambda x: np.isneginf(x).any())].tolist()
print(f"Columns with -inf values: {cols_with_pos_inf}")

Columns with -inf values: ['bjets_dPhi_llr', 'bjets_dEta_llr', 'bjets_pt_bb_llr', 'bjet0_pt_llr', 'bjets_mean_pt_llr', 'trijet_pt_llr', 'trijet_pt_rat_llr', 'trijet_bijet_dR_llr', 'trijet_bijet_dPhi_llr', 'trijet_bijet_dEta_llr', 'bjet_bijet_dR_llr', 'bjet_bijet_dPhi_llr', 'bjet_bijet_dEta_llr', 'blnu_mT_llr', 'blnu_bl_mInv_llr', 'blnu_lnu_mT_llr', 'all_sT_llr', 'all_mInv_llr', 'all_mT_llr', 'all_jets_HT_llr', 'all_pt_llr', 'WW_mInv_llr', 'WW_pt_llr', 'jj_l_dPhi_llr', 'jj_l_dR_llr', 'jj_lnu_dPhi_llr', 'jj_lnu_dR_llr', 'bb_lnu_dPhi_llr', 'bb_lnu_dR_llr', 'min_b_l_dPhi_llr', 'min_b_lnu_dPhi_llr', 'lep0_pt_llr', 'lep0_iso_llr', 'ak4_jet0_pt_llr', 'ak4_jet0_eta_llr', 'ak4_jet0_phi_llr', 'ak4_jet1_pt_llr', 'ak4_jet1_eta_llr', 'ak4_jet1_phi_llr', 'ak4_jet2_pt_llr', 'ak4_jet2_eta_llr', 'ak4_jet2_phi_llr', 'ak4_jet3_pt_llr', 'ak4_jet3_eta_llr', 'ak4_jet3_phi_llr', 'met_pt_llr', 'met_phi_llr', 'nAK4_llr', 'nAK4_btag_llr']


In [149]:
df_sub = df[['event', 'genWeight', 'trijet_pt_rat', 'trijet_pt_rat_llr']]
nan_rows = df_sub[df_sub.isna().any(axis=1)]
inf_rows = df_sub[np.isinf(df_sub).any(axis=1)]
inf_rows

,event,genWeight,trijet_pt_rat,trijet_pt_rat_llr
118,17919,0.033119,0.999487,-inf
224,8859,0.033119,0.993043,-inf
439,23179,0.033119,0.992481,-inf
441,25563,0.033119,0.994416,-inf
627,80081,0.033119,0.993666,-inf
979,81897,0.033119,0.992849,-inf
1011,81913,0.033119,0.997010,-inf
1182,38797,0.033119,0.993659,-inf
1293,20809,0.033119,0.997743,-inf
1528,49827,0.033119,0.995843,-inf


In [150]:
inf_rows['genWeight'].sum()

0.39743283

In [151]:
nan_rows

,event,genWeight,trijet_pt_rat,trijet_pt_rat_llr
225,8875,0.033119,0.989409,NaN
478,23447,0.033119,0.989441,NaN
1234,2213,0.033119,0.990636,NaN


In [152]:
nan_rows['genWeight'].sum()

0.0993582

In [162]:
with open('llrs_vars1D.json', 'r') as f:
    file_llr_func = json.load(f)
    corrs = file_llr_func['corrections']
    met_pt_llr_func = corrs[0]
    trijet_pt_llr_func = corrs[1]

def get_bin_content(value, name):
    if name=='met_pt_llr': corr = met_pt_llr_func
    elif name=='trijet_pt_rat_llr': corr = trijet_pt_llr_func
    edges = corr['data']['edges']
    content = corr['data']['content']
    bin_index = np.digitize(value, edges) - 1
    # Check bounds
    if bin_index < 0 or bin_index >= len(content):
        return None
    return content[bin_index]

In [164]:
print(get_bin_content(0.992625, 'trijet_pt_rat_llr'))

-inf


In [135]:
import ROOT
tfile = ROOT.TFile.Open(str(root_file))
df = ROOT.RDataFrame("SL_4j_resolved", tfile)
x = "met_pt_llr"

# A) Counts of special values (no C++ helper redeclarations needed)
n_all = int(df.Count().GetValue())
n_nan = int(df.Filter(f"TMath::IsNaN({x})").Count().GetValue())
# "Finite" in ROOT means not NaN and not ±Inf
n_fin = int(df.Filter(f"TMath::Finite({x})").Count().GetValue())
n_inf = n_all - n_fin - n_nan                      # total ±Inf
# Optional split of ±Inf into signs (requires <cmath> if you want std::isinf)
# Here’s a TMath-only equivalent:
n_pos_inf = int(df.Filter(f"!TMath::Finite({x}) && !TMath::IsNaN({x}) && {x} > 0").Count().GetValue())
n_neg_inf = n_inf - n_pos_inf
print(f"all={n_all}  NaN={n_nan}  Inf={n_inf}  +Inf={n_pos_inf}  -Inf={n_neg_inf}")

# B) Fix a single binning and reuse it
nbins, xmin, xmax = 100, -3, 3

# Unweighted, all rows
h_all = df.Histo1D(("h_all","",nbins,xmin,xmax), x).GetValue()
nb = h_all.GetNbinsX()
print(f"[ALL rows] Entries(set by RDF)={h_all.GetEntries():.0f}  "
      f"Underflow(count)={h_all.GetBinContent(0):.0f}  Overflow(count)={h_all.GetBinContent(nb+1):.0f}")

all=1783  NaN=1  Inf=4  +Inf=0  -Inf=4
[ALL rows] Entries(set by RDF)=1783  Underflow(count)=4  Overflow(count)=1
